In [ ]:
import os
import cv2
import numpy as np
from ctypes import POINTER, cast
from comtypes import CLSCTX_ALL
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

WIDTH, HEIGHT = 1100, 700

script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
def Webcam(cameras : int):
    if type(cameras) == int and camera:
        return cv2.VideoCapture(cameras)
    
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Access Webcam")

devices = AudioUtilities.GetSpeakers()

interface = devices.Activate(
    IAudioEndpointVolume._iid_,
    CLSCTX_ALL,
    None
)

volume_control = cast(
    interface,
    POINTER(IAudioEndpointVolume)
)

MIN_VOLUME = 0
MAX_VOLUME = 100

MIN_RATIO = 1.05
MAX_RATIO = 1.65

current_volume = int(
    volume_control.GetMasterVolumeLevelScalar() * 100
)

smoothed_volume = current_volume

while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_skin = np.array([0, 30, 60])
    upper_skin = np.array([25, 180, 255])

    mask = cv2.inRange(hsv, lower_skin, upper_skin)

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask, cv2.MORPH_OPEN, kernel
    )

    mask = cv2.morphologyEx(
        mask, cv2.MORPH_CLOSE, kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:

        largest_contour = max(
            contours,
            key=cv2.contourArea
        )

        area = cv2.contourArea(largest_contour)

        if area > 5000:

            hull = cv2.convexHull(largest_contour)

            hull_area = cv2.contourArea(hull)

            if area > 0:

                spread_ratio = hull_area / area

                spread_ratio = np.clip(
                    spread_ratio,
                    MIN_RATIO,
                    MAX_RATIO
                )

                target_volume = np.interp(
                    spread_ratio,
                    [MIN_RATIO, MAX_RATIO],
                    [MIN_VOLUME, MAX_VOLUME]
                )

                smoothed_volume = (
                    smoothed_volume * 0.85
                    + target_volume * 0.15
                )

                current_volume = int(smoothed_volume)

                volume_control.SetMasterVolumeLevelScalar(
                    current_volume / 100,
                    None
                )

            cv2.drawContours(
                frame,
                [largest_contour],
                -1,
                (0, 255, 0),
                2
            )

            cv2.drawContours(
                frame,
                [hull],
                -1,
                (0, 0, 255),
                2
            )

    frame = cv2.resize(frame, (WIDTH, HEIGHT))

    cv2.rectangle(
        frame,
        (WIDTH - 250, 20),
        (WIDTH - 20, 100),
        (255, 255, 255),
        -1
    )

    cv2.rectangle(
        frame,
        (WIDTH - 250, 20),
        (WIDTH - 20, 100),
        (0, 0, 0),
        3
    )

    cv2.putText(
        frame,
        f"Volume : {current_volume} %",
        (WIDTH - 240, 72),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 0),
        2
    )

    cv2.imshow("Gesture Volume Control", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

camera.release()
cv2.destroyAllWindows()